<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/vision-language-composition/blob/main/scripts/MACCO_Train_Short_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MACCO: two-epoch training run

**Goal:** run the authors' `src/training/main_train_macco_clip.py` on 128 prepared COCO rows, save its checkpoint, and understand what ran. Use a Colab **GPU**. Run cells from top to bottom. No Google Drive and no evaluation datasets are needed.

This is a small **execution check**. Its losses and benchmark scores are not comparable to the paper's five-epoch, ~110k-pair training run. The notebook fetches only the required `src/` tree and prepared TSV from [the authors' repository](https://github.com/hiker-lw/MACCO). It stores everything temporarily under `/content/macco_run/`.

### 1. Install training imports

Colab supplies PyTorch and torchvision. Install the remaining packages imported by this training path; we do not install the authors' entire pinned requirements file.

In [1]:
%pip -q install "timm==1.0.15" ftfy regex "webdataset==0.2.111" braceexpand easydict PyYAML pytz wandb huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00


### 2. Get the necessary author files

This is a partial Git checkout: `src/` plus the prepared COCO training TSV. It does **not** download the full author repository or their benchmark datasets.

In [2]:
from pathlib import Path
import subprocess
import torch

ROOT = Path('/content/macco_run')
TSV = ROOT / 'mscoco_object_relation_attribute_with_phrase_bbox.tsv'
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', '--depth=1',
                    'https://github.com/hiker-lw/MACCO.git', str(ROOT)], check=True)
if not (ROOT / 'src/training/main_train_macco_clip.py').is_file():
    subprocess.run(['git', '-C', str(ROOT), 'sparse-checkout', 'set', '--no-cone',
                    '/src/', '/mscoco_object_relation_attribute_with_phrase_bbox.tsv'], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', 'HEAD'], check=True)
assert TSV.is_file() and (ROOT / 'src/config/MACCO_CLIP.yaml').is_file()
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not found')
assert torch.cuda.is_available(), 'Choose a GPU runtime in Colab first.'
print('Training annotation:', TSV, f'({TSV.stat().st_size / 1e6:.1f} MB)')

GPU: Tesla T4
Training annotation: /content/macco_run/mscoco_object_relation_attribute_with_phrase_bbox.tsv (47.9 MB)


### 3. Select a small training subset

The original TSV has captions and precomputed object, relation, attribute, and box information. The model's data loader expects these columns, so we keep each selected row intact.

In [3]:
import pandas as pd

data = pd.read_csv(TSV, sep='\t')
print('Original rows:', len(data))
print('Columns:', list(data.columns))
assert {'filepath', 'title', 'relation_attribute_phrase_bbox_list'}.issubset(data.columns)
small = data[data.filepath.astype(str).str.contains(
    r'COCO_train2014_\d+\.jpg', regex=True, na=False
)].head(128).copy()
assert len(small) == 128, 'Expected 128 train2014 rows in the author TSV.'
small_path = ROOT / 'small_mscoco.tsv'
small.to_csv(small_path, sep='\t', index=False)
print('Selected rows:', len(small), '| unique images:', small.filepath.nunique())
display(small[['filepath', 'title']].head(3))

Original rows: 109468
Columns: ['title', 'filepath', 'neg_caption', 'neg_image', 'object_list', 'relation_list', 'attribute_list', 'relation_phrase_list', 'attribute_phrase_list', 'relation_attribute_phrase_bbox_list']
Selected rows: 128 | unique images: 128


,filepath,title
29427,coco/train2014/COCO_train2014_000000057870.jpg,A table is adorned with wooden chairs with blu...
29428,coco/train2014/COCO_train2014_000000222016.jpg,A man using a phone in a phone booth.
29429,coco/train2014/COCO_train2014_000000520950.jpg,A VERY SMALL KITCHEN WITH A STOVE AND A SHELF ...


### 4. Obtain only the images named in the subset

The author repository contains the TSV but **not the COCO images**. This cell fetches matching `train2014` filenames from a [COCO train2014 image mirror](https://huggingface.co/datasets/ali-sh07/COCO-train2014/tree/main). It checks that each downloaded file opens as an image. If the mirror lacks a filename, get that original COCO image and put it under the exact path printed by the error. Already downloaded images are reused on rerun.

In [5]:
import pandas as pd
from huggingface_hub import HfApi

data = pd.read_csv(TSV, sep="\t")

available = {
Path(name).name
for name in HfApi().list_repo_files(
repo_id="ali-sh07/COCO-train2014",
repo_type="dataset"
)
if name.endswith(".jpg")
}

matching = data[
data.filepath.astype(str).map(
lambda name: Path(name).name in available
)
]
small = matching.head(128).copy()

assert len(small) == 128, f"Only {len(small)} matching rows found."

small_path = ROOT / "small_mscoco.tsv"
small.to_csv(small_path, sep="\t", index=False)

print("Selected rows:", len(small))
print("Images to download:", small.filepath.nunique())

Selected rows: 128
Images to download: 128


### 5. Check one prepared example

In [6]:
from ast import literal_eval

example = small.iloc[0]
image_path = ROOT / 'datasets' / str(example['filepath'])
with Image.open(image_path) as im:
    print('Image:', image_path.name, '| size:', im.size)
print('Caption:', example['title'])
for name in ('object_list', 'relation_list', 'attribute_list',
             'relation_attribute_phrase_bbox_list'):
    print(name, ':', str(literal_eval(example[name]))[:180])

Image: COCO_train2014_000000057870.jpg | size: (640, 480)
Caption: A table is adorned with wooden chairs with blue accents.
object_list : ['chairs', 'accents', 'table']
relation_list : []
attribute_list : ['blue', 'wooden']
relation_attribute_phrase_bbox_list : [{'blue accents': [[89, 263, 257, 476]], 'wooden chairs': [[2, 253, 637, 475]]}]


### 6. Configure the **author** training script

The script reads `src/config/MACCO_CLIP.yaml` after parsing command-line options. We change the local copy of that YAML. The model and four loss weights stay as released. `--no-val` skips the compositional benchmarks, which are separate from this training check.

In [7]:
import yaml

config_path = ROOT / 'src/config/MACCO_CLIP.yaml'
config = yaml.safe_load(config_path.read_text())
config.update(train_data='small_mscoco.tsv', epochs=2, batch_size=16,
              workers=2, warmup=5, save_frequency=1,
              logs='logs/colab_short', report_to='')
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
for key in ('train_data', 'model', 'pretrained', 'epochs', 'batch_size',
            'workers', 'warmup', 'contrast_loss_w', 'intra_modal_contrast_loss_w',
            'mask_text_loss_w', 'mask_image_loss_w'):
    print(f'{key}: {config[key]}')

train_data: small_mscoco.tsv
model: ViT-B-32
pretrained: openai
epochs: 2
batch_size: 16
workers: 2
warmup: 5
contrast_loss_w: 1.0
intra_modal_contrast_loss_w: 0.1
mask_text_loss_w: 20.0
mask_image_loss_w: 250.0


In [15]:

from pathlib import Path

path = ROOT / "src/open_clip_code/MACCO_variant/MACCO_text_image.py"
code = path.read_text()

for method, target in [
    ("clip_encode_text", "self.CLIP"),
    ("clip_encode_image", "self.CLIP"),
    ("clip_vit_forward", "self.CLIP.visual"),
    ("clip_forward", "self.CLIP"),
]:
    old = f"self.{method}.__get__({target})"
    new = f"type(self).{method}.__get__({target}, type({target}))"

    assert old in code or new in code, f"Could not find {method}"

    code = code.replace(old, new)

path.write_text(code)
print("Method bindings fixed.")


Method bindings fixed.


### 7. Train for two epochs

This runs the actual author file. The model starts from OpenAI CLIP weights, **not** the authors' fully trained MACCO checkpoint. The first run may take time to download those initial weights.

In [ ]:
subprocess.run(['python', 'src/training/main_train_macco_clip.py', '--no-val'],
               cwd=ROOT, check=True)

### 8. Save the checkpoint and log

Run this after training finishes. Download both before the Colab runtime ends. The checkpoint holds the CLIP weights; the author's save code does not include predictor or optimizer state for a faithful resume.

In [ ]:
checkpoints = sorted(ROOT.glob('logs/colab_short/**/checkpoints/epoch_2.pt'),
                     key=lambda p: p.stat().st_mtime)
assert checkpoints, 'No epoch_2.pt found; inspect the training error above.'
checkpoint = checkpoints[-1]
log = checkpoint.parent.parent / 'out.log'
print('Checkpoint:', checkpoint, f'({checkpoint.stat().st_size / 1e6:.1f} MB)')
print('Log:', log, '| exists:', log.is_file())
from google.colab import files
files.download(str(checkpoint))
if log.is_file():
    files.download(str(log))